In [ ]:
%load_ext dotenv
%dotenv ../../05_src/.secrets

In [ ]:
import os
import chromadb
from chromadb.utils.embedding_functions import OpenAIEmbeddingFunction

def search_chromadb(prompt, n=5, collection_name="my_collection", persist_directory="./chromadb"):
    """
    Search a persistent ChromaDB collection for similar documents.
    
    Args:
        prompt (str): The search query
        n (int): Number of results to return (default: 5)
        collection_name (str): Name of the collection (default: "my_collection")
        persist_directory (str): Path to the ChromaDB storage directory
    
    Returns:
        dict: Search results containing documents, distances, and metadata
    """
    # Initialize the persistent client
    chroma_client = chromadb.PersistentClient(path=persist_directory)

    # Debug purposes
    # print(chroma_client.heartbeat())
    # # Get all collections
    # collections = chroma_client.list_collections()
    # # Print collection names
    # print("Available collections:")
    # for collection in collections:
    #     print(f"  - {collection.name}")
    #     print(f"    Documents: {collection.count()}")
    #     print(f"    Metadata: {collection.metadata}")
    #     print()
    
    # Initialize the same embedding function you used when creating the collection
    embedding_function = OpenAIEmbeddingFunction(
        api_key=os.getenv("OPENAI_API_KEY"),
        model_name="text-embedding-3-small"
    )
    
    # Get the existing collection
    collection = chroma_client.get_collection(
        name=collection_name,
        embedding_function=embedding_function
    )
    
    # Perform the similarity search
    results = collection.query(
        query_texts=[prompt],
        n_results=n
    )
    
    return results


if __name__ == "__main__":
    # Example usage
    query = "For BANCO SANTANDER MÉXICO, what was the gross operating income for 4Q22?"
    num_results = 5
    
    # Perform the search
    results = search_chromadb(prompt=query, n=num_results)
    
    # Display results
    print(f"Search query: {query}")
    print(f"Found {len(results['documents'][0])} results:\n")
    
    for i, (doc, distance, doc_id) in enumerate(zip(
        results['documents'][0], 
        results['distances'][0],
        results['ids'][0]
    ), 1):
        print(f"Result {i} (ID: {doc_id}, Distance: {distance:.4f}):")
        print(f"{doc[:200]}...")  # Print first 200 characters
        print("-" * 80)

Search query: For BANCO SANTANDER MÉXICO, what was the gross operating income for 4Q22?
Found 5 results:

Result 1 (ID: 50409_66154_37577, Distance: 0.3540):
|:---------------------------|----:|
| Earnings Release | 4Q.2021 | nan |
| Banco Santander México     | nan |
| nan                        |   6 |
Gross operating income
Banco Santander México’s gros...
--------------------------------------------------------------------------------
Result 2 (ID: 53979_66161_56937, Distance: 0.3743):
Gross operating income
Banco Santander México’s gross operating
income for 2Q22 totaled Ps.23,619 million, representing increases of 10.1% YoY, or Ps.2,159 million, and 5.8% QoQ, or Ps.1,288 million.
...
--------------------------------------------------------------------------------
Result 3 (ID: 59683_66176_33316, Distance: 0.3828):
Gross operating income
Banco Santander México’s gross operating
income for 1Q23 totaled Ps.26,116 million, representing increases of 16.9% YoY, and 0.1% QoQ. The YoY in

Examples of queries that gave wrong answers:

"How much were the Mexican Government contributions in PETRÓLEOS MEXICANOS as of March 31, 2023?"

"For BANCO SANTANDER MÉXICO, regarding Deposits, what was the December 2022 YoY increase?"

"For BANCO SANTANDER MÉXICO, what was the gross operating income for 4Q22?"